In [ ]:
# Parameters

param_intermediate_location = "/home/jovyan/Cloud Storage/naa-vre-user-data"
param_model_type = 'multiple' #multiple, default, min, max, calibrated (for now)
conf_tmp_data = "/tmp/data"

In [ ]:
# Dummy cell - links to Loobos FLM Landis-ii runner
loobos_scenario_datafile <- ""

In [ ]:
# Parameter sweep 
# ==================================================

print(paste0(conf_tmp_data, "/", loobos_scenario_datafile))
# Create time-stamped folder to store model runs & their inputs & outputs
datetime_str <- format(Sys.time(), "%Y%m%d-%H%M")
run_dir <- file.path(param_intermediate_location, paste0("workflow_run_", datetime_str))
writeLines(datetime_str, file.path(conf_tmp_data, "datetime_str.txt"))


dir.create(file.path(run_dir, "input"), recursive = TRUE)
dir.create(file.path(run_dir, "outputs"), recursive = TRUE)

# Copy all created input files from the working directory to the input directory
input_dir <- file.path(run_dir, "input")

list_of_files <- c(
    "Preprocessed_KNMI_monthly.csv", "Preprocessed_KNMI_spinup_monthly.csv", "loobos_spp_ecoregion_data.csv", 
    "loobos_species_data.csv", "loobos_core_species_data.txt", "loobos_initial_communities.csv", "loobos_initial_communities.tif",
    "loobos_ecoregions.txt", "loobos_ecoregions.tif",
    "loobos_biomass_succession.txt", "loobos_biomass_output.txt", 
    "loobos_scenario.txt", "loobos_climate_generator.txt"
)

from_files <- file.path(conf_tmp_data, list_of_files)

to_files <- file.path(input_dir, list_of_files)

file.copy(from = file.path(conf_tmp_data, list_of_files),
          to = to_files, 
          overwrite = TRUE)

#Define parameters
parameters <- tibble::tribble(
  ~"parameter",        ~"species",            ~"initial",   ~"min",   ~"max",    ~"calibrated",
  # --- ANPPMax ---
  "ANPPMax",         "betula",                 1130,         548,      1520,      548,
  "ANPPMax",         "pinussylvestris",        712,          551,      1375,      551,
  "ANPPMax",         "quercus",                1130,         548,      1392,      548,
  "ANPPMax",         "pseudotsugamenziesii",   712,          551,      1865,      551,
  # --- BiomassMax ---
  "BiomassMax",      "betula",                 17740,        6270,     15150,     27300,
  "BiomassMax",      "pinussylvestris",        17740,        6540,     14390,     27300,
  "BiomassMax",      "quercus",                17740,        6270,     15150,     27300,
  "BiomassMax",      "pseudotsugamenziesii",   17740,        6540,     14390,     27300,
  # --- GrowthCurve ---
  "GrowthCurve",     "betula",                 0.74,         0.2,      0.87,      0.2,
  "GrowthCurve",     "pinussylvestris",        0.74,         0.62,     1.0,       0.62,
  "GrowthCurve",     "quercus",                0.74,         0.4,      0.92,      0.4,
  "GrowthCurve",     "pseudotsugamenziesii",   0.74,         0.2,      0.87,      0.2,
  # --- MortalityCurve ---
  "MortalityCurve",  "betula",                 12,            5,        15,        12,
  "MortalityCurve",  "pinussylvestris",        12,            9,        25,        12,
  "MortalityCurve",  "quercus",                12,            9,        25,        12,
  "MortalityCurve",  "pseudotsugamenziesii",   12,            9,        25,        12,
  # --- Longevity ---
  "Longevity",       "betula",                 220,          90,       150,       150,
  "Longevity",       "pinussylvestris",        1000,         400,      750,       750,
  "Longevity",       "quercus",                1400,         500,      1000,      1000,
  "Longevity",       "pseudotsugamenziesii",   1400,         500,      1000,      1000,
  # --- ProbEstablish ---
  "ProbEstablish",   "betula",                 1.0,          0.3,      0.5,       0.3,
  "ProbEstablish",   "pinussylvestris",        1.0,          0.5,      0.5,       0.5,
  "ProbEstablish",   "quercus",                0.1,          0.1,      0.3,       0.1,
  "ProbEstablish",   "pseudotsugamenziesii",   1.0,          0.3,      0.5,       0.3
)
 
if (param_model_type == 'multiple') {
  
  # ========== FULL PARAMETER SWEEP ==========
  message("Running full parameter sweep")
  
  # Generate all 3^6 = 729 parameter combinations
  groups <- c("initial", "min", "max")
  param_grid <- expand.grid(
    ANPPMax = groups,
    BiomassMax = groups,
    GrowthCurve = groups,
    MortalityCurve = groups,
    Longevity = groups,
    ProbEstablish = groups,
    stringsAsFactors = FALSE
  )
  
  parameter_set_numbers <- sprintf("%03d", 1:nrow(param_grid))
  
  for (i in 1:nrow(param_grid)) {
    combo <- param_grid[i, ]
    
    # Extract parameter set for this combination
    parameter_set <- parameters %>%
      tidyr::pivot_longer(cols = c("initial", "min", "max", "calibrated"),
                          names_to = "group", values_to = "value") %>%
      dplyr::filter(
        (.data$parameter == "ANPPMax" & .data$group == combo$ANPPMax) |
        (.data$parameter == "BiomassMax" & .data$group == combo$BiomassMax) |
        (.data$parameter == "GrowthCurve" & .data$group == combo$GrowthCurve) |
        (.data$parameter == "MortalityCurve" & .data$group == combo$MortalityCurve) |
        (.data$parameter == "Longevity" & .data$group == combo$Longevity) |
        (.data$parameter == "ProbEstablish" & .data$group == combo$ProbEstablish)
      ) %>%
      dplyr::select("parameter", "species", "value", "group") 
    
    # Save CSV
    readr::write_csv(parameter_set,
                     file.path(run_dir, "input", paste0("params_", parameter_set_numbers[i], ".csv")))
  }
 
} else if (param_model_type == 'calibrated') {
  
  # ========== CALIBRATED PARAMETERS ==========
  message("Running calibrated parameter combination")
  
  # Filter for calibrated values
  parameter_set <- parameters %>%
    tidyr::pivot_longer(cols = c("initial", "min", "max", "calibrated"), 
                        names_to = "group", values_to = "value") %>%
    dplyr::filter(.data$group == 'calibrated') %>%
    dplyr::select("parameter", "species", "value", "group")
  
  parameter_set_numbers <- "001"
  readr::write_csv(parameter_set, 
                   file.path(run_dir, "input", "params_001.csv"))
 
} else if (param_model_type == 'initial') {
  
  # ========== INITIAL PARAMETERS ==========
  message("Running initial parameter combination")
  
  # Filter for initial values
  parameter_set <- parameters %>%
    tidyr::pivot_longer(cols = c("initial", "min", "max", "calibrated"), 
                        names_to = "group", values_to = "value") %>%
    dplyr::filter(.data$group == 'initial') %>%
    dplyr::select("parameter", "species", "value", "group")
  
  parameter_set_numbers <- "002"
  readr::write_csv(parameter_set, 
                   file.path(run_dir, "input", "params_002.csv"))

} else if (param_model_type == 'min') {
  
  # ========== MINIMUM PARAMETERS ==========
  message("Running minimum parameter combination")
  
  # Filter for min values
  parameter_set <- parameters %>%
    tidyr::pivot_longer(cols = c("initial", "min", "max", "calibrated"), 
                        names_to = "group", values_to = "value") %>%
    dplyr::filter(.data$group == 'min') %>%
    dplyr::select("parameter", "species", "value", "group")
  
  parameter_set_numbers <- "003"
  readr::write_csv(parameter_set, 
                   file.path(run_dir, "input", "params_003.csv"))

} else if (param_model_type == 'max') {
  
  # ========== MAXIMUM PARAMETERS ==========
  message("Running maximum parameter combination")
  
  # Filter for max values
  parameter_set <- parameters %>%
    tidyr::pivot_longer(cols = c("initial", "min", "max", "calibrated"), 
                        names_to = "group", values_to = "value") %>%
    dplyr::filter(.data$group == 'max') %>%
    dplyr::select("parameter", "species", "value", "group")
  
  parameter_set_numbers <- "004"
  readr::write_csv(parameter_set, 
                   file.path(run_dir, "input", "params_004.csv"))
    
} else {
  stop(glue::glue("Unknown param_model_type: {param_model_type}. Must be 'multiple', 'calibrated', 'initial', 'min', or 'max'"))
}

# ======================================
# Modify input files
# ========================================

for (n_parameter_set in parameter_set_numbers) {

  # ========== Read the parameter set for this run ==========
  parameter_set <- readr::read_csv(
    file.path(run_dir, "input", paste0("params_", n_parameter_set, ".csv")),
    show_col_types = FALSE
  )
  
  # ========== Update species ecoregion file ==========
  spp_params <- parameter_set %>%
    dplyr::filter(.data$parameter %in% c("ANPPMax", "BiomassMax", "ProbEstablish"))
  
  # Read from working directory
  spp_ecoregion <- readr::read_csv(
    file.path(input_dir, "loobos_spp_ecoregion_data.csv"), #read from input directory
    show_col_types = FALSE
  )
  
  # Convert parameter set to wide format (remove group column)
  spp_params_wide <- spp_params %>%
    dplyr::select(-"group") %>%
    tidyr::pivot_wider(names_from = "parameter", values_from = "value") 
  
  # Join and keep only needed columns
  spp_ecoregion_clean <- spp_ecoregion %>%
    dplyr::select("SpeciesCode", "Year", "EcoregionName", "ProbMortality") %>%
    dplyr::inner_join(spp_params_wide, 
                      by = c("SpeciesCode" = "species"))
  
  readr::write_csv(spp_ecoregion_clean, paste0(conf_tmp_data,"/", "loobos_spp_ecoregion_data.csv")) #write to tmp data
    
  # ========== Update species data file ==========
  curve_params <- parameter_set %>%
    dplyr::filter(.data$parameter %in% c("GrowthCurve", "MortalityCurve"))

    
  # Read the original file
  curves <- readr::read_csv(
    file.path(input_dir, "loobos_species_data.csv"),
    show_col_types = FALSE
  )


  # Convert to wide format (remove group column)
  curve_params_wide <- curve_params %>%
    dplyr::select(-"group") %>%
    tidyr::pivot_wider(names_from = "parameter", values_from = "value") 
  
  # Join
  curves_clean <- curves %>%
    dplyr::select(-any_of(c("GrowthCurve", "MortalityCurve"))) %>%
    dplyr::inner_join(curve_params_wide,
                      by = c("SpeciesCode" = "species"))

  if (nrow(curves_clean) == 0) {
  stop("ERROR: Join produced 0 rows - species codes don't match!")
}
  
  readr::write_csv(curves_clean, paste0(conf_tmp_data, "/", "loobos_species_data.csv"))
  
  # ========== Update core species data file ==========
longevity_params <- parameter_set %>%
  dplyr::filter(.data$parameter == "Longevity") 
 
core_species_file <- file.path(input_dir, "loobos_core_species_data.txt")
 
# Read the file, keeping header lines separate
all_lines <- readLines(core_species_file)
header <- all_lines[1:2]
data_lines <- all_lines[3:length(all_lines)]
 
# Parse data as tab-separated
core_species_data <- read.table(
  text = data_lines,
  sep = "\t",
  stringsAsFactors = FALSE
)
 
colnames(core_species_data) <- c(
  "SpeciesCode", "Longevity", "Sexual_Maturity", "Eff_Seed_Disp_Dist", "Max_Seed_Disp_Dist", 
  "Veg_Preprod_Prob", "Min_Resprout_Age", "Max_Resprout_Age", "Postfire_Regen")
 
# Create lookup table
longevity_lookup <- setNames(longevity_params$value, longevity_params$species)
 
# Update Longevity column
core_species_data$Longevity <- longevity_lookup[core_species_data$SpeciesCode]

# Convert back to tab-separated format
data_output <- apply(core_species_data, 1, function(row) paste(row, collapse = "\t"))
output_lines <- c(header, data_output)
writeLines(output_lines, paste0(conf_tmp_data, "/", "loobos_core_species_data.txt"))

                     }
writeLines(parameter_set_numbers, 
           file.path(conf_tmp_data, "parameter_set_numbers.txt"))

In [ ]:
print(parameter_set_numbers)